genera un script python completo y explicado usando el dataset '../data/breast_cancer.csv' para ejemplificar el uso de t-SNE, incluye los siguientes puntos: - estandarización de los datos - análisis de dimensionalidad - reducción de dimensionalidad - análisis de clusterización - clusterización con k-means - clusterización con Gaussian Mixture - Genera 3 graficos en 1 columna para comparar clustrización kmeans, Gaussian Mixture y Segmentos reales. Si hay valores nulos realiza imputaciones con medias. Muestra el dataset antes y después de establecer t-SNE.

In [1]:
# Script/Cell completo y explicado (en español) para ejemplificar t-SNE + clustering
# Usa el dataset '../data/breast_cancer.csv'
# Puntos cubiertos:
# - carga e inspección inicial (muestra dataset antes de t-SNE)
# - imputación de valores nulos con la media
# - estandarización de los datos
# - análisis de dimensionalidad con PCA (varianza explicada)
# - reducción de dimensionalidad con PCA + t-SNE
# - análisis y ejecución de clusterización (KMeans y Gaussian Mixture)
# - gráficos comparativos (3 filas x 1 columna): KMeans, GMM, etiquetas reales (si existen)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

sns.set(style="whitegrid")
RANDOM_STATE = 42

# 1) Cargar el dataset
path = '../data/breast_cancer.csv'
df = pd.read_csv(path)
print("Dataset cargado. Dimensiones:", df.shape)
print("\nPrimeras filas (antes de t-SNE):")
display(df.head())

# 2) Identificar columna de etiqueta (si existe)
# Buscamos nombres comunes; si no hay etiqueta real, seguiremos sin ella.
possible_label_cols = ['target', 'diagnosis', 'label', 'Class', 'diagnosis_result', 'diagnosis_label']
label_col = None
for col in possible_label_cols:
    if col in df.columns:
        label_col = col
        break

# Si no encontramos pero hay una columna con solo dos valores 'M'/'B' u '0'/'1', la detectamos:
if label_col is None:
    for col in df.columns:
        if df[col].nunique() <= 3:  # candidate small-cardinality column
            unique_vals = set(df[col].dropna().unique())
            if unique_vals <= set(['M','B']) or unique_vals <= set([0,1]) or unique_vals <= set(['malignant','benign']):
                label_col = col
                break

# Separar features y etiqueta (si hay)
if label_col:
    y = df[label_col].copy()
    X = df.drop(columns=[label_col]).copy()
    # Si la etiqueta no es numérica, la codificamos
    if y.dtype == object or y.dtype.name == 'category':
        y = LabelEncoder().fit_transform(y.astype(str))
    print(f"\nEtiqueta real detectada en columna: '{label_col}'. Usaremos esa como 'y'.")
else:
    y = None
    X = df.copy()
    print("\nNo se detectó una columna de etiqueta real. 'Segmentos reales' en los gráficos será una etiqueta ficticia si es necesario.")

# 3) Manejo de valores nulos: imputación por la media para variables numéricas
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) == 0:
    raise ValueError("No se encontraron columnas numéricas en las que trabajar.")
imputer = SimpleImputer(strategy='mean')
X[numeric_cols] = imputer.fit_transform(X[numeric_cols])

# Si existen columnas no numéricas, las intentamos convertir o descartamos (aquí convertimos booleanas/categoricas simples)
non_numeric = [c for c in X.columns if c not in numeric_cols]
if non_numeric:
    # Intentamos una conversión sencilla: si son categóricas, las codificamos
    for c in non_numeric:
        try:
            X[c] = pd.to_numeric(X[c])
            numeric_cols.append(c)
        except Exception:
            # Para columnas no convertibles, las descartamos para este ejemplo numérico
            X = X.drop(columns=[c])
            print(f"Columna '{c}' descartada (no numérica y no convertible).")

print("\nValores nulos tras imputación (por columna):")
print(X.isnull().sum())

# 4) Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nEstandarización realizada. Forma de X_scaled:", X_scaled.shape)

# 5) Análisis de dimensionalidad con PCA
# Calculamos PCA completo para ver la varianza explicada acumulada
pca_full = PCA(n_components=min(X_scaled.shape[1], X_scaled.shape[0]), random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
explained_ratio = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained_ratio)

# Determinar nº de componentes para explicar 95% de la varianza
n_components_95 = np.searchsorted(cumulative, 0.95) + 1
print(f"\nComponentes necesarios para explicar 95% de la varianza: {n_components_95}")

# Gráfico de varianza explicada acumulada
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(cumulative)+1), cumulative, marker='o')
plt.axhline(0.95, color='r', linestyle='--', label='95% umbral')
plt.xlabel('Número de componentes principales')
plt.ylabel('Varianza explicada acumulada')
plt.title('Análisis de dimensionalidad (PCA)')
plt.legend()
plt.tight_layout()
plt.show()

# 6) Reducción de dimensionalidad
# Buenas prácticas: reducir con PCA a un número moderado (ej. 50 o n_components_95), luego aplicar t-SNE
pca_for_tsne = min(50, X_scaled.shape[1])  # to speed up t-SNE
pca = PCA(n_components=pca_for_tsne, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print("\nForma de X tras PCA previa a t-SNE:", X_pca.shape)

# Aplicar t-SNE a 2 dimensiones para visualización
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, n_iter=1000, random_state=RANDOM_STATE, init='pca')
X_tsne = tsne.fit_transform(X_pca)

# Mostrar dataset (o al menos una vista) después de t-SNE
df_tsne = pd.DataFrame(X_tsne, columns=['TSNE-1', 'TSNE-2'])
print("\nDataset reducido con t-SNE (primeras filas):")
display(df_tsne.head())

# 7) Clusterización
# Ejecución de KMeans y Gaussian Mixture. En este ejemplo usaremos k=2..4 y escogeremos k=2 (común en cáncer de mama benign/mal).
k = 2
kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE)
labels_kmeans = kmeans.fit_predict(X_pca)

gmm = GaussianMixture(n_components=k, random_state=RANDOM_STATE)
labels_gmm = gmm.fit_predict(X_pca)

# Opcional: calcular silhouette score sobre los datos usados para clustering (PCA space)
sil_km = silhouette_score(X_pca, labels_kmeans) if len(set(labels_kmeans)) > 1 else np.nan
sil_gmm = silhouette_score(X_pca, labels_gmm) if len(set(labels_gmm)) > 1 else np.nan
print(f"\nSilhouette score (KMeans): {sil_km:.4f}")
print(f"Silhouette score (GMM): {sil_gmm:.4f}")

# Preparar la etiqueta "real" para graficar; si no hay etiqueta, usamos 0.. (etiqueta ficticia)
if y is not None:
    true_labels = np.asarray(y)
    true_title = f"Segmentos reales ('{label_col}')"
else:
    true_labels = np.zeros(X.shape[0], dtype=int)
    true_title = "Segmentos reales (no hay etiqueta en el dataset)"

# 8) Graficar 3 gráficos en 1 columna para comparar kmeans, GMM y segmentos reales
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(6, 15), sharex=True, sharey=True)

# Colormap consistente
palette = sns.color_palette("tab10", n_colors=max(k, len(np.unique(true_labels))))

# KMeans plot
ax = axes[0]
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=labels_kmeans, palette=palette, legend='full', ax=ax, s=50)
ax.set_title("KMeans (labels sobre t-SNE)")
ax.set_xlabel("TSNE-1")
ax.set_ylabel("TSNE-2")
ax.legend(title='KMeans', loc='best')

# GMM plot
ax = axes[1]
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=labels_gmm, palette=palette, legend='full', ax=ax, s=50)
ax.set_title("Gaussian Mixture (labels sobre t-SNE)")
ax.set_xlabel("TSNE-1")
ax.set_ylabel("TSNE-2")
ax.legend(title='GMM', loc='best')

# Real segments plot
ax = axes[2]
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=true_labels, palette=palette, legend='full', ax=ax, s=50)
ax.set_title(true_title)
ax.set_xlabel("TSNE-1")
ax.set_ylabel("TSNE-2")
ax.legend(title='Real', loc='best')

plt.tight_layout()
plt.show()

# 9) Resumen final
print("\nResumen:")
print("- Dataset original mostrado al inicio (primeras filas).")
print("- t-SNE aplicado y mostrado (primeras filas del dataframe con columnas 'TSNE-1' y 'TSNE-2').")
print(f"- KMeans y GMM ejecutados con k={k}. Silhouette KMeans={sil_km:.4f}, GMM={sil_gmm:.4f}.")
print("- Recomendación: para análisis cuantitativos use clustering en el espacio de características (o PCA), y reserve t-SNE solo para visualización. Evite sacar conclusiones de clusters únicamente basadas en t-SNE.")

# (Opcional) Si quieres guardar los embeddings y etiquetas en csv:
# df_out = pd.concat([df.reset_index(drop=True), df_tsne, pd.DataFrame({'kmeans': labels_kmeans, 'gmm': labels_gmm})], axis=1)
# df_out.to_csv('../data/breast_cancer_tsne_clustered.csv', index=False)

Dataset cargado. Dimensiones: (569, 33)

Primeras filas (antes de t-SNE):


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN



Etiqueta real detectada en columna: 'diagnosis'. Usaremos esa como 'y'.


/home/patricioaraneda/Documentos/github/Cienciadedatos/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['Unnamed: 32']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


ValueError: Columns must be same length as key